# FedHDPrivacy — a walkthrough

Companion notebook for *Privacy-Preserving Federated Learning with Differentially Private
Hyperdimensional Computing* (Computers and Electrical Engineering, 2025).

This notebook walks through the framework piece by piece: the hyperdimensional encoder, the
noise accounting, and then a full federated run. It calls the installed package rather than
redefining anything, so what you see here is exactly what `fedhdprivacy` runs.

**Setup** — from the repository root:

```bash
pip install -e ".[all]"
```

**A note on the data.** The paper's CNC machining dataset is proprietary and not included.
This notebook uses UCI HAR, which downloads itself on first use: smartphone accelerometer and
gyroscope features from 30 subjects, split so that each client holds a distinct group of
subjects. Set `DATASET = "synthetic"` below to run entirely offline.

In [ ]:
import logging

import matplotlib.pyplot as plt
import numpy as np
import torch

import fedhdprivacy as fhp
from fedhdprivacy.utils import configure_logging

configure_logging(logging.WARNING)   # keep the notebook quiet
print("fedhdprivacy", fhp.__version__, "| torch", torch.__version__)

DATASET = "uci_har"   # "uci_har" | "mnist" | "fashion_mnist" | "synthetic"

---
## 1. The data

`load_dataset` downloads the data, scales features into `[0, 1]`, and splits it across clients.
The `natural` partition keeps each subject's samples together, so the heterogeneity comes from
who generated the data rather than from an artificial sampling scheme.

In [ ]:
dataset = fhp.load_dataset(DATASET, n_clients=8, partition="natural", seed=42)
print(dataset.summary())

for client in dataset.clients[:3]:
    counts = np.bincount(client.y, minlength=dataset.n_classes)
    print(f"{client.client_id}: {len(client):5d} samples, class counts {counts}")

### How non-IID is it?

Each row is a client, each column a class. Uniform rows would mean an IID split; the variation
here is what makes federated aggregation non-trivial.

In [ ]:
matrix = np.stack([
    np.bincount(c.y, minlength=dataset.n_classes) / len(c) for c in dataset.clients
])

fig, ax = plt.subplots(figsize=(7, 4))
image = ax.imshow(matrix, cmap="viridis", aspect="auto")
ax.set_xlabel("Class")
ax.set_ylabel("Client")
ax.set_yticks(range(dataset.n_clients), [c.client_id for c in dataset.clients])
ax.set_title("Class distribution per client")
fig.colorbar(image, ax=ax, label="Fraction of client's data")
plt.tight_layout()
plt.show()

---
## 2. The encoder

Encoding maps a feature vector to a bipolar hypervector in `{-1, +1}^D` through a random
projection. The property that makes hyperdimensional classification work is **locality**:
similar inputs must produce similar hypervectors. Let's check that directly.

In [ ]:
encoder = fhp.Encoder(
    n_features=dataset.n_features,
    dimensions=10000,
    backend="projection",   # "density" reproduces the original notebook's torchhd encoder
    seed=42,
)

sample = torch.from_numpy(dataset.clients[0].x[:1])
print("input :", tuple(sample.shape))
print("output:", tuple(encoder(sample).shape), "values in", encoder(sample).unique().tolist())

In [ ]:
# Perturb one sample by increasing amounts and watch the similarity decay.
anchor = dataset.clients[0].x[0]
deltas = np.linspace(0, 0.5, 25)
rng = np.random.default_rng(0)
direction = rng.normal(size=anchor.shape).astype(np.float32)
direction /= np.linalg.norm(direction)

variants = np.clip(anchor + np.outer(deltas, direction), 0, 1).astype(np.float32)
encoded = encoder(torch.from_numpy(variants))
similarity = (encoded @ encoded[0]) / encoder.dimensions

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(deltas, similarity.numpy(), marker="o")
ax.set_xlabel("Perturbation size in input space")
ax.set_ylabel("Hypervector similarity")
ax.set_title("Encoding preserves locality")
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

---
## 3. The noise accounting

This is the heart of the paper. At each round a client works out:

- **required** — the noise privacy demands given every sample seen so far,
- **inherited** — the noise the downloaded global model already carries,
- **added** — the difference, which is all it actually injects.

None of this needs training data, so we can plot the whole schedule instantly.

In [ ]:
D, EPS, K, L = 10000, 10.0, 8, 1000
kwargs = dict(dimensions=D, epsilon=EPS, n_clients=K, samples_per_round=L)

rounds = np.arange(1, 51)
required   = [fhp.required_noise_variance(**kwargs, round_index=r) for r in rounds]
inherited  = [fhp.cumulative_noise_variance(**kwargs, round_index=r) for r in rounds]
added      = [fhp.additional_noise_variance(**kwargs, round_index=r) for r in rounds]

fig, (left, right) = plt.subplots(1, 2, figsize=(13, 4.5))

left.plot(rounds, required,  marker="s", markevery=5, label=r"required  $\xi^r_k$")
left.plot(rounds, added,     marker="o", markevery=5, ls="--", label=r"added  $\Gamma^r_k$")
left.plot(rounds, inherited, marker="^", markevery=5, alpha=0.7, label=r"inherited  $\Psi^{r-1}_k$")
left.set_xlabel("Communication round"); left.set_ylabel("Noise variance")
left.set_title("Adaptive noise schedule"); left.legend(); left.grid(alpha=0.3)

saved = 100 * (1 - np.array(added) / np.array(required))
right.plot(rounds, saved, color="crimson", marker="o", markevery=5)
right.set_xlabel("Communication round"); right.set_ylabel("Noise avoided (%)")
right.set_title("Savings from accounting for inherited noise"); right.grid(alpha=0.3)

plt.tight_layout(); plt.show()
print(f"After 50 rounds the client injects {100 - saved[-1]:.1f}% of the naive requirement.")

The gap on the left is the contribution. A framework that re-noised from scratch each round
would follow the top curve forever; FedHDPrivacy follows the dashed one.

### Why the server adds nothing

Averaging `K` independently noised models divides the noise variance by `K`, but the
sensitivity of the aggregate also falls by `K`. Theorems 3 and 5 show the first effect wins:
`γ = Var(inherited) / Var(required) > 1` always, so the aggregate is already private.

In [ ]:
for k in (2, 5, 10, 50, 100):
    gamma = (k * np.log(1.25 * L)) / np.log(1.25 * k * L)
    print(f"K = {k:3d}   gamma = {gamma:6.2f}   {'private' if gamma > 1 else 'NOT private'}")

---
## 4. A full federated run

Ten communication rounds, eight clients, `D = 10000`, `ε = 10`.

Each round every client receives the global model, retrains it on a fresh shard of `L` samples,
adds the incremental noise, and sends it back. The server averages. Note that **round 1 lands
near chance** — the model has seen very little data and carries the full noise burden with
nothing inherited to offset it.

In [ ]:
config = fhp.ExperimentConfig(
    dataset=DATASET,
    n_clients=8,
    rounds=10,
    dimensions=10000,
    local_epochs=10,
    epsilon=10.0,
    encoder="projection",
    seed=42,
)

configure_logging(logging.INFO)
history = fhp.run_federated_training(dataset, config)
configure_logging(logging.WARNING)

print()
print(history.final_report)

In [ ]:
fig, (left, right) = plt.subplots(1, 2, figsize=(13, 4.5))

left.plot(range(1, len(history.accuracies) + 1),
          [a * 100 for a in history.accuracies], marker="o", color="crimson")
left.axhline(100 / dataset.n_classes, ls=":", color="gray", label="chance")
left.set_xlabel("Communication round"); left.set_ylabel("Global accuracy (%)")
left.set_title("Accuracy improves despite repeated noising"); left.legend(); left.grid(alpha=0.3)

rs = [r.round_index for r in history.rounds]
right.plot(rs, [r.noise_required for r in history.rounds], marker="s", label="required")
right.plot(rs, [r.noise_added for r in history.rounds], marker="o", ls="--", label="added")
right.set_xlabel("Communication round"); right.set_ylabel("Noise variance")
right.set_title("Noise injected during this run"); right.legend(); right.grid(alpha=0.3)

plt.tight_layout(); plt.show()

---
## 5. What privacy costs

Rerunning with DP disabled gives the accuracy ceiling. The gap is the price of the guarantee.

In [ ]:
baseline_config = fhp.ExperimentConfig(**{**config.to_dict(), "differential_privacy": False})
baseline = fhp.run_federated_training(dataset, baseline_config, progress=False)

print(f"with DP (eps={config.epsilon:g}): {history.final_report.accuracy * 100:.2f}%")
print(f"without DP:                {baseline.final_report.accuracy * 100:.2f}%")
print(f"cost of privacy:           {(baseline.final_report.accuracy - history.final_report.accuracy) * 100:.2f} points")

---
## 6. Sweeping the privacy budget

Noise variance scales as `1/eps^2`, so tightening the budget gets expensive quickly. This cell
takes a few minutes; reduce `dimensions` or `rounds` to speed it up.

In [ ]:
budgets = [4, 6, 8, 10, 20]
sweep = {}

for eps in budgets:
    cfg = fhp.ExperimentConfig(**{**config.to_dict(), "epsilon": eps, "dimensions": 5000})
    sweep[eps] = fhp.run_federated_training(dataset, cfg, progress=False)
    print(f"eps = {eps:5.1f}  ->  accuracy {sweep[eps].final_report.accuracy * 100:5.2f}%")

In [ ]:
fig, (left, right) = plt.subplots(1, 2, figsize=(13, 4.5))

for eps, h in sweep.items():
    left.plot(range(1, len(h.accuracies) + 1), [a * 100 for a in h.accuracies],
              marker="o", label=fr"$\epsilon$ = {eps:g}")
left.set_xlabel("Communication round"); left.set_ylabel("Accuracy (%)")
left.set_title("Tighter budgets learn more slowly"); left.legend(); left.grid(alpha=0.3)

right.plot(budgets, [sweep[e].final_report.accuracy * 100 for e in budgets],
           marker="o", color="darkgreen")
right.set_xlabel(r"Privacy budget $\epsilon$"); right.set_ylabel("Final accuracy (%)")
right.set_title("More privacy, less accuracy"); right.grid(alpha=0.3)

plt.tight_layout(); plt.show()

---
## 7. Where to go next

- `fedhdprivacy --help` — every option from the command line
- `configs/` — ready-made configurations
- `scripts/sweep_privacy_budget.py` — the `D` x `eps` grid from Figure 10
- `docs/ALGORITHM.md` — the derivations behind `privacy.py`
- `docs/REPRODUCING.md` — what does and does not carry over from the paper

Directions the paper flags as open: defences against model poisoning and free-riding,
multimodal data fusion, deployment on edge hardware, and per-client privacy budgets that adapt
to how sensitive each client's data is.